In [ ]:
import os
from glob import glob

import numpy as np
import torch
from matplotlib import pyplot as plt
from tifffile import imread
from torchvision.transforms import v2
from torchvision.tv_tensors import Image, Mask


class SparseLabeledImageDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        base_path,
        transforms=None,
        image_subfolder="tif",
        label_subfolder="labels_sparse",
        min_labeled_pixels=500,
    ):
        """
        PyTorch Dataset of sparsely labelled (0 considered unlabelled) TIF stacks.
        Will only include planes with a minimum number of nonzero pixels.
        
        Parameters
        ----------
        base_path: str
            path of the dataset
        image_subfolder: str
            subfolder containing images in TIF format
        label_subfolder: str
            subfolder containing integer label maps (0: unlabelled, 1,2,...: labelled)
        transforms: torchvision v2 transform
            transforms to be applied to images. should only be for augmentation
            conversion to tensor is done already
        min_labeled_pixels: int
            minimum number of labeled pixels for a plane to be included in the dataset
        """

        self.images = []
        self.masks = []
        self.transforms = transforms

        # NOTE: we assume matched images and masks in same lexicographical order
        img_files = sorted(glob(os.path.join(base_path, image_subfolder, "*.tif")))
        mask_files = sorted(glob(os.path.join(base_path, label_subfolder, "*.tif")))

        for img_file, mask_file in zip(img_files, mask_files):

            img = imread(img_file)
            mask = imread(mask_file)

            # add dummy z axis for 2D data
            if img.ndim == 2:
                img = img[np.newaxis]
                mask = mask[np.newaxis]
            
            img_selected, mask_selected = get_labeled_planes(
                img, mask, min_labeled_pixels
            )

            # to torch tensors with standard datatypes
            self.images.extend(torch.from_numpy(img_selected).float())
            self.masks.extend(torch.from_numpy(mask_selected).long())

        # convert to torchvision TVTensors (so augmentations can be easily applied to both img and mask)
        self.images = [Image(img) for img in self.images]
        self.masks = [Mask(mask) for mask in self.masks]

    def __getitem__(self, idx):

        img, mask = self.images[idx], self.masks[idx]

        if self.transforms is not None:
            img, mask = self.transforms(img, mask)

        return img, mask

    def __len__(self):

        return len(self.images)


def get_labeled_planes(img, mask, min_labeled_pixels=1):
    """
    Get the subset of xy planes from pairs of images and label maps in which there are at least min_labeled_pixels with nonzero label.
    The last two dimensions are interpreted as yx, the result will have shape (N_labeled_planes, Y, X).
    """

    # sum binarized mask over last 2 dimensions, get selection of planes with enough labeled pixels
    mask_bin = mask > 0
    selection = (
        mask_bin.sum(axis=tuple(range(mask.ndim - 2, mask.ndim))) >= min_labeled_pixels
    )

    return img[selection], mask[selection]


In [ ]:
from unet import LightningUNet
from lightning import pytorch as L
from torch.nn.functional import cross_entropy


def get_masked_loss_input(logits_pred, y_gt):

    """
    Get loss function input for sparse segmentation.
    Will select from predicted logits and ground-truth labels 
    """

    # mask == 0 indicates unlabelled
    selection = y_gt > 0

    # select logits -> (npix, c) array 
    logits_selected = torch.transpose(logits_pred, 0, 1)[:, selection].T

    # select nonzero from gt mask, subtract 1 (label==1 in sparse indicates background==0, 2 indicates 1, ...)
    y_selected_corrected = y_gt[selection] - 1
    
    return logits_selected, y_selected_corrected



class SparseSegmentationUNet(LightningUNet):

    """
    Training subclass of ligthning UNet for sparse labels
    """

    def training_step(self, batch, batch_idx):
        
        # apply net
        x, y = batch
        yp = self.forward(x)

        # select pixels labeled in GT, calculate CE for those
        logits_selected, y_selected = get_masked_loss_input(yp, y)
        loss = cross_entropy(logits_selected, y_selected)

        self.log('train_loss', loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)



In [ ]:
base_dir = '/Users/david/Desktop/jurkat_nucleolin/'

# random crop and flips
tr = v2.Compose(
    [
        v2.RandomCrop((256, 256)),
        v2.RandomHorizontalFlip(),
        v2.RandomVerticalFlip()
    ]
)

dataset = SparseLabeledImageDataset(base_dir, transforms=tr)

In [ ]:
img, mask = dataset[51]

fig, axs = plt.subplots(ncols=2)
axs[0].imshow(img.squeeze())
axs[1].imshow(mask.squeeze())


len(dataset)

In [ ]:
from torch.utils.data import DataLoader

net = SparseSegmentationUNet(3, [24, 48, 96])
loader = DataLoader(dataset, batch_size=8, shuffle=True)

trainer = L.Trainer(
    logger=L.loggers.CSVLogger(""),
    log_every_n_steps=np.ceil(len(dataset) / loader.batch_size),
    max_epochs=300,
)

trainer.fit(net, loader)

In [ ]:
from lightning.pytorch.utilities.model_summary import ModelSummary

net_inference =  LightningUNet.load_from_checkpoint('/Users/david/Desktop/jurkat_nucleolin/unet_nucleolus_001/checkpoints/epoch=299-step=3300.ckpt').eval()
ModelSummary(net_inference, max_depth=3)

In [ ]:
test_file = '/Volumes/nn/Julia Vogtmann/Microscopy/26JV_018/tif/0001_ch1.tif'

# add two dummy dimensions (batch size, channels)
img = torch.from_numpy(imread(test_file)).float()[:, torch.newaxis, torch.newaxis]

# ALTERNATIVE with full loader (different batch size, etc.):
# img = torch.from_numpy(imread(test_file)).float()[:, torch.newaxis]
# predict_ds = torch.utils.data.TensorDataset(img)
# predict_loader = torch.utils.data.DataLoader(predict_ds, 1)


trainer = L.Trainer(enable_checkpointing=False, logger=False)
with torch.no_grad():
    pred = trainer.predict(net_inference, img)
    pred = torch.concat(pred)
    pred_labels = pred.argmax(1)

In [ ]:
import napari

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.Viewer()
viewer.add_image(img.squeeze())

# view predictions of a single class
viewer.add_labels((pred_labels==2).int())

# ALTERNATIVE: all classes
# viewer.add_labels(pred_labels)